# FoundationPose real sobre una pieza *text-to-CAD*

**TFM — Estimación de pose 6-DoF para bin picking.** Este cuaderno ejecuta la **red neuronal real**
[NVlabs/FoundationPose](https://github.com/NVlabs/FoundationPose) (CVPR 2024) sobre la escuadra en L
generada con *text-to-CAD* (exp27), usando la **imagen RGBD real** capturada en CoppeliaSim.

Cierra el último eslabón que no puede correr en el MacBook M1 (la red exige CUDA): aquí, con la
**GPU de Colab**, se estima la pose de verdad y se compara con la *ground-truth* del simulador.

> **Requisito:** Entorno de ejecución con GPU (`Entorno de ejecución → Cambiar tipo → GPU T4`).


## 1. Comprobar GPU


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout.split('\n')[0:10])
assert torch.cuda.is_available(), 'Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU'
print('CUDA OK:', torch.cuda.get_device_name(0))


## 2. Clonar FoundationPose y construir dependencias

El repositorio incluye extensiones CUDA que se compilan en el primer arranque (varios minutos).


In [ ]:
%cd /content
![ -d FoundationPose ] || git clone https://github.com/NVlabs/FoundationPose.git
%cd /content/FoundationPose
# dependencias principales (Colab ya trae torch/cuda)
!pip install -q trimesh open3d pyrender opencv-python-headless scipy numpy imageio Pillow
!pip install -q git+https://github.com/NVlabs/nvdiffrast.git
# Kaolin y extensiones (según instrucciones del repo):
!pip install -q kaolin==0.15.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.1.0_cu121.html || echo 'ajusta la versión de kaolin a tu torch/cuda'
# construir las extensiones C++/CUDA de FoundationPose
!bash build_all_conda.sh 2>/dev/null || python -c "import os; os.system('cd bundlesdf/mycuda && pip install -e .')"
print('build intentado — revisa mensajes; puede requerir ajustar versiones a tu runtime')


## 3. Descargar los pesos preentrenados

Descarga los pesos del refiner y del scorer desde el enlace oficial de FoundationPose
(ver README del repo). Colócalos en `/content/FoundationPose/weights/`.

```
weights/
  2023-10-28-18-33-37/   # refiner
  2024-01-11-20-02-45/   # scorer
```


In [ ]:
import os
os.makedirs('/content/FoundationPose/weights', exist_ok=True)
# Opción con gdown si tienes el ID de la carpeta de Drive de los pesos:
# !pip install -q gdown && gdown --folder <ID_DRIVE_PESOS> -O /content/FoundationPose/weights/
print('Pesos esperados en:', sorted(os.listdir('/content/FoundationPose/weights')))


## 4. Traer los datos del TFM (mesh + RGBD real capturada)

Se clona el repositorio público del TFM para obtener la pieza generada y la imagen RGBD real
capturada en CoppeliaSim (experimento exp27).


In [ ]:
%cd /content
![ -d tfm ] || git clone https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git tfm
EXP = '/content/tfm/experiments/results/exp27_text_to_cad'
import numpy as np, json, trimesh, os
mesh_file = f'{EXP}/assets/test_bracket.obj'
rgb = np.load(f'{EXP}/figs/e2e_rgb.npy')            # (H,W,3) uint8, real
depth = np.load(f'{EXP}/figs/e2e_depth.npy')        # (H,W) float, metros (near/far reales)
report = json.load(open(f'{EXP}/e2e_report.json'))
print('rgb', rgb.shape, 'depth', depth.shape, depth.min().round(3), depth.max().round(3), 'm')
print('GT xyz (m):', report['pose']['gt_xyz_m'])


## 5. Preparar entradas para FoundationPose

FoundationPose recibe `rgb`, `depth` (m), `mask` del objeto y la matriz intrínseca `K`.

- **Máscara:** el objeto se pintó de rojo en la escena → segmentación por color (equivale al
  *mask* que entregaría un detector).
- **Convención de imagen:** la cámara de CoppeliaSim entrega la imagen con una rotación de 180°
  respecto a la convención de pinhole estándar; se corrige con `np.rot90(x, 2)` para que `K` aplique.
- **K:** sensor 640×480, FOV 1.047 rad → fx = fy = 554.3, cx = 320, cy = 240.


In [ ]:
import numpy as np
# rotar 180° para la convención estándar (ver exp27: validado contra GT a 0.5 cm)
rgb2 = np.rot90(rgb, 2).copy()
depth2 = np.rot90(depth, 2).copy().astype(np.float32)
r, g, b = rgb2[:,:,0].astype(int), rgb2[:,:,1].astype(int), rgb2[:,:,2].astype(int)
mask = ((r > 110) & (g < 90) & (b < 90)).astype(np.uint8)
K = np.array([[554.3, 0, 320.0],[0, 554.3, 240.0],[0, 0, 1.0]], dtype=np.float32)
print('máscara:', int(mask.sum()), 'px del objeto')
assert mask.sum() > 100, 'máscara vacía: revisa el umbral de color'


## 6. Ejecutar FoundationPose (registro model-based)

Se carga el mesh (en metros) y se llama a `register()` con la RGBD real. Devuelve la pose
objeto→cámara (matriz 4×4 en SE(3)).


In [ ]:
%cd /content/FoundationPose
import sys; sys.path.insert(0, '/content/FoundationPose')
import trimesh, numpy as np
from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor
import dr  # nvdiffrast context helper del repo (o usar RasterizeCudaContext)
mesh = trimesh.load('/content/tfm/experiments/results/exp27_text_to_cad/assets/test_bracket.obj')
mesh.apply_scale(0.001)  # mm -> m
scorer = ScorePredictor(); refiner = PoseRefinePredictor()
glctx = dr.RasterizeCudaContext()
est = FoundationPose(model_pts=mesh.vertices, model_normals=mesh.vertex_normals,
                     mesh=mesh, scorer=scorer, refiner=refiner, glctx=glctx)
pose_cam = est.register(K=K, rgb=rgb2, depth=depth2, ob_mask=mask, iteration=5)
print('pose objeto→cámara (4x4):\n', np.round(pose_cam, 4))


## 7. Comparar con la *ground-truth* del simulador

La GT del simulador está en el mundo. Se transforma la pose de FoundationPose (en cámara) al mundo
usando la pose conocida de la cámara `T_wc`, y se compara la traslación con la GT del reporte.


In [ ]:
import numpy as np, json
rep = json.load(open('/content/tfm/experiments/results/exp27_text_to_cad/e2e_report.json'))
gt = np.array(rep['pose']['gt_xyz_m'])
# cámara fija en la escena bin_base: pos [0.26,0.10,1.30], R=diag(1,-1,-1)
T_wc = np.array([[1,0,0,0.26],[0,-1,0,0.10],[0,0,-1,1.30],[0,0,0,1.0]])
pose_world = T_wc @ pose_cam
t_est = pose_world[:3,3]
t_err_mm = np.linalg.norm(t_est - gt) * 1000
print('pose estimada (mundo) xyz:', np.round(t_est,4).tolist())
print('ground-truth        xyz:', np.round(gt,4).tolist())
print(f'--> error de traslación FoundationPose REAL: {t_err_mm:.1f} mm')
print('Compara con el proxy clásico local del TFM (exp27): ~4 mm.')


## 8. (Opcional) Visualizar el overlay

Proyecta el modelo en la pose estimada sobre la imagen para inspección visual.


In [ ]:
# from Utils import draw_posed_3d_box, draw_xyz_axis  # helpers del repo
# vis = draw_xyz_axis(rgb2.copy(), ob_in_cam=pose_cam, scale=0.05, K=K)
# import matplotlib.pyplot as plt; plt.figure(figsize=(6,5)); plt.imshow(vis); plt.axis('off'); plt.show()
print('Descomenta si los helpers Utils del repo están disponibles.')


---
### Notas honestas
- Este cuaderno ejecuta la **red neuronal real** de FoundationPose; el TFM en local usa un **proxy
  clásico** (FPFH+ICP) porque el M1 no tiene CUDA. Ambos parten de la **misma RGBD real** del simulador.
- Las versiones de `kaolin`/`nvdiffrast`/extensiones CUDA deben casar con el `torch`+CUDA del runtime
  de Colab del día; si el build falla, ajústalas según el README de FoundationPose (es el paso frágil).
- La pieza tiene geometría de *ground-truth exacta* (CAD generado), por lo que el error de pose es
  medible sin etiquetado manual.
